# Online learning: hands-on

Run each cell with **Shift + Enter**.

## 1. Training one row at a time with `partial_fit`

Most scikit-learn models have a `fit` method, which trains on **all** the data at once and starts from scratch every time.
Some models also have `partial_fit`, which **continues** training from where the model left off. That is online learning.
`SGDRegressor` is one of them: it does linear regression (covered later in the course), but learns step by step.

In [ ]:
import time
import numpy as np
from sklearn.linear_model import SGDRegressor

model = SGDRegressor()

# One row of data: 3 input columns and 1 output value
X1 = np.array([[1.0, 2.0, 3.0]])
y1 = np.array([10.0])

start = time.perf_counter()
model.partial_fit(X1, y1)              # learn from this single row
print(f"Trained on 1 row in {time.perf_counter() - start:.5f} seconds")

# A new row arrives later: keep training from where we stopped
X2 = np.array([[2.0, 1.0, 0.5]])
y2 = np.array([6.0])
model.partial_fit(X2, y2)              # the model is updated, not rebuilt
print("Coefficients after 2 rows:", model.coef_)

## 2. When the world changes: batch vs online

The data below follows one rule for the first 500 points (*y = 2x*), then the rule flips (*y = 8 - 2x*).

- The **batch** model is trained once on the first 300 points, then frozen.
- The **online** model learns from every mini-batch of 10 points as it arrives.

The chart shows each model's error over time (lower is better). **Try:** move the learning-rate slider.
A low rate adapts slowly after the change. A higher rate recovers much faster, but its error before the change is a little higher, because it reacts more to noise.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, SGDRegressor

rng = np.random.default_rng(0)
n, change = 1000, 500
x = rng.uniform(0, 4, n)
y = np.where(np.arange(n) < change, 2 * x, 8 - 2 * x) + rng.normal(0, 0.5, n)
X = x.reshape(-1, 1)
batches = range(0, n, 10)                                   # mini-batches of 10 rows

batch_model = LinearRegression().fit(X[:300], y[:300])     # trained once, then frozen

def errors(model_predict, update=None):
    out = []
    for b in batches:
        Xb, yb = X[b:b + 10], y[b:b + 10]
        out.append(np.mean(np.abs(model_predict(Xb) - yb)))  # error on data it has not seen yet
        if update:
            update(Xb, yb)                                  # then learn from it
    return out

rates = [0.001, 0.01, 0.1]
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(batches), y=errors(batch_model.predict), name="batch (frozen)",
                         line=dict(color="#4C78A8", width=3)))
for i, eta in enumerate(rates):
    online = SGDRegressor(learning_rate="constant", eta0=eta, random_state=0)
    online.partial_fit(X[:10], y[:10])
    fig.add_trace(go.Scatter(x=list(batches), y=errors(online.predict, online.partial_fit), visible=(i == 1),
                             name=f"online (learning rate {eta})", line=dict(color="#54A24B", width=3)))

steps = [dict(method="update", label=str(eta), args=[{"visible": [True] + [j == i for j in range(len(rates))]}])
         for i, eta in enumerate(rates)]
fig.add_vline(x=change, line_dash="dot", annotation_text="the rule changes")
fig.update_layout(sliders=[dict(active=1, steps=steps, currentvalue=dict(prefix="Online learning rate: "))],
                  xaxis_title="Data points seen", yaxis_title="Error", height=520, template="simple_white")
fig.show()